# Week 5: Predictive Modeling and Algorithm Selection
## Customer Trend Prediction in the Beauty & Wellness Industry

**Objective:** Compare a simple baseline with Ridge Regression and a shallow Decision Tree Regressor to estimate daily service demand.

**Important limitation:** The source is synthetic and contains only 100 appointments over one month. Results are an educational demonstration, not evidence of real-world predictive performance. The dataset does not contain appointment status, so cancellation prediction is not possible with these columns.

## 1. Import libraries and load the dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

df = pd.read_csv("salon_appointments_dummy.csv")
df.head()

## 2. Data quality and preparation

In [ ]:
print("Dataset shape:", df.shape)
print("\\nColumns:", df.columns.tolist())
print("\\nMissing values:\\n", df.isna().sum())
print("\\nDuplicate rows:", df.duplicated().sum())
print("Duplicate appointment IDs:", df["appointment_id"].duplicated().sum())

df["appointment_date"] = pd.to_datetime(
    df["appointment_date"], dayfirst=True, errors="coerce"
)
df["appointment_time"] = pd.to_datetime(
    df["appointment_time"], format="%H:%M", errors="coerce"
)
df = df.dropna(subset=["appointment_date", "appointment_time", "service_name"]).copy()
df["weekday"] = df["appointment_date"].dt.dayofweek
df["hour"] = df["appointment_time"].dt.hour

print("Cleaned shape:", df.shape)
print("Date range:", df["appointment_date"].min().date(), "to", df["appointment_date"].max().date())
print("Unique customers:", df["customer_id"].nunique())
print("Unique services:", df["service_name"].nunique())

## 3. Exploratory analysis

In [ ]:
service_counts = df["service_name"].value_counts().sort_values(ascending=False)
display(service_counts.rename("appointment_count").to_frame())

service_counts.plot(kind="bar", figsize=(8, 4), title="Appointments by Service")
plt.xlabel("Service")
plt.ylabel("Appointments")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

weekday_counts = df.groupby(df["appointment_date"].dt.day_name()).size()
weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
weekday_counts = weekday_counts.reindex(weekday_order, fill_value=0)
display(weekday_counts.rename("appointment_count").to_frame())

weekday_counts.plot(kind="bar", figsize=(8, 4), title="Appointments by Weekday")
plt.xlabel("Weekday")
plt.ylabel("Appointments")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()

## 4. Create a service-by-date target table
Each row represents one service on one calendar date. Missing service-date combinations are filled with zero bookings so the model can distinguish a zero-booking day from a missing row.

In [ ]:
dates = pd.date_range(df["appointment_date"].min(), df["appointment_date"].max(), freq="D")
services = sorted(df["service_name"].unique())

grid = pd.MultiIndex.from_product(
    [dates, services], names=["appointment_date", "service_name"]
).to_frame(index=False)

counts = (
    df.groupby(["appointment_date", "service_name"])
      .size()
      .rename("booking_count")
      .reset_index()
)

daily = grid.merge(counts, on=["appointment_date", "service_name"], how="left")
daily["booking_count"] = daily["booking_count"].fillna(0).astype(int)
daily["weekday"] = daily["appointment_date"].dt.dayofweek

display(daily.head(10))
print("Rows in modeling table:", len(daily))

## 5. Chronological train/test split
The earliest approximately 80% of dates are used for training and the latest dates for testing. This is more appropriate than a random split for a time-related problem, though the short date range makes the test metrics uncertain.

In [ ]:
unique_dates = sorted(daily["appointment_date"].unique())
split_idx = max(1, int(len(unique_dates) * 0.8))
cutoff = pd.Timestamp(unique_dates[split_idx - 1])

train = daily[daily["appointment_date"] <= cutoff].copy()
test = daily[daily["appointment_date"] > cutoff].copy()

features = ["service_name", "weekday"]
X_train, y_train = train[features], train["booking_count"]
X_test, y_test = test[features], test["booking_count"]

print("Training dates:", train["appointment_date"].min().date(), "to", train["appointment_date"].max().date())
print("Testing dates:", test["appointment_date"].min().date(), "to", test["appointment_date"].max().date())
print("Training rows:", len(train), "| Testing rows:", len(test))

## 6. Train and compare models
Metrics: MAE (average absolute error in bookings), RMSE (penalizes large errors), and R² (variance-relative score; it may be negative on a test set).

In [ ]:
preprocess = ColumnTransformer([
    ("service", OneHotEncoder(handle_unknown="ignore"), ["service_name"])
], remainder="passthrough")

models = {
    "Mean baseline": DummyRegressor(strategy="mean"),
    "Ridge regression": Ridge(alpha=1.0),
    "Decision tree (depth=3)": DecisionTreeRegressor(
        max_depth=3, min_samples_leaf=3, random_state=42
    )
}

results = []
for name, estimator in models.items():
    pipe = Pipeline([("preprocess", preprocess), ("model", estimator)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    results.append({
        "Model": name,
        "MAE": mean_absolute_error(y_test, pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, pred)),
        "R2": r2_score(y_test, pred)
    })

metrics = pd.DataFrame(results).round(3)
display(metrics)

## 7. Interpret the results
Compare each candidate with the baseline. A lower MAE/RMSE is generally better; a higher R² is generally better, but R² alone is not enough. Do not claim that a model is production-ready from this single small chronological holdout. If the candidate models do not consistently beat the baseline on later periods, retain the baseline and collect more data.

## 8. Business interpretation and limitations
- Forecasts could support staff and appointment-slot planning after further validation.
- Booking count is not the same as profit; costs and service duration are not modeled here.
- The dataset is synthetic and spans only one month, so it cannot establish annual seasonality.
- No appointment-status field is present, so cancellation classification cannot be trained.
- Collect several months of data, include zero-booking dates, and evaluate using rolling chronological windows before deployment.